# Backend de Visión Artificial BJJ: Fusión 3D con YOLO26 (Google Colab Pro)

Este notebook implementa el servidor de inferencia remota con aceleración GPU para el proyecto **Asistente de Visión Artificial BJJ**, cumpliendo estrictamente con los requisitos **RD-01** y **RF-03** de la investigación:

- **RD-01:** Utilización de la suite YOLO26 (`yolo26x-pose.pt` para 17 articulaciones COCO y `yolo26x-depth.pt` para profundidad métrica real).
- **RF-03:** Fusión geométrica sincrónica en $\mathbb{R}^3$, donde $Z = \text{depth\_map}[y, x]$ en metros reales absolutos, sin requerir sensores RGB-D activos.
- **Optimización Colab Pro:** Gestión proactiva de VRAM en GPUs A100/T4 (`torch.cuda.empty_cache()`) y persistencia del túnel seguro Ngrok.

### Celda 1: Diagnóstico de Hardware y Memoria GPU (A100 / T4)

In [1]:
# Celda 1: Diagnóstico riguroso de GPU y RAM disponible
import torch
import psutil

assert torch.cuda.is_available(), "ERROR: No se detectó GPU. Activa T4 o A100 en Entorno de ejecución > Cambiar tipo de entorno de ejecución."

ram_gb = psutil.virtual_memory().total / 1e9
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
gpu_name = torch.cuda.get_device_name(0)

print(f"--- Estado del Entorno de Ejecución ---")
print(f"✅ GPU Detectada: {gpu_name}")
print(f"✅ VRAM Dedicada: {vram_gb:.2f} GB")
print(f"✅ RAM de Sistema: {ram_gb:.2f} GB")
print(f"✅ PyTorch Version: {torch.__version__} | CUDA: {torch.version.cuda}")

### Celda 2: Instalación de Dependencias

In [ ]:
# Celda 2: Instalación de YOLO26, Flask, Ngrok y dependencias de visión
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "ultralytics>=8.4.0", "sentence-transformers", "accelerate", "flask", "pyngrok", "opencv-python-headless", "numpy", "psutil"
])
print("✅ Dependencias instaladas correctamente.")

### Celda 3: Carga de Modelos YOLO26x en VRAM (Requisito RD-01)

In [ ]:
# Celda 3: Carga de modelos YOLO26x según RD-01 con gestión de memoria
from ultralytics import YOLO
import torch

# Liberar memoria residual en GPU
torch.cuda.empty_cache()

print("Cargando YOLO26x-Pose (Keypoints 2D COCO en VRAM)...\n")
model_pose = YOLO("yolo26x-pose.pt")
model_pose.to("cuda")

print("Cargando YOLO26x-depth (Profundidad Métrica en VRAM)...\n")
model_depth = YOLO("yolo26x-depth.pt")
model_depth.to("cuda")

print("✅ Modelos YOLO26 cargados exitosamente en GPU.")

### Celda 4: Inferencia y Fusión Geométrica 3D Métrica (Requisito RF-03)

In [ ]:
# Celda 4: Pipeline de Fusión 3D (RF-03) con Extracción y Anotación de Fotograma Real en GPU
import os
import cv2
import base64
import tempfile
import numpy as np
import torch

def procesar_video_y_fusionar_3d(video_bytes: bytes) -> dict:
    """
    Ejecuta YOLO26x-Pose y YOLO26x-depth de forma sincrónica en GPU.
    Asigna a cada articulación 2D (x, y) su coordenada Z métrica en metros reales.
    Captura el fotograma real con OpenCV, anota los keypoints y lo retorna en base64.
    """
    torch.cuda.empty_cache()
    tmp = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False)
    try:
        tmp.write(video_bytes)
        tmp.flush()
        tmp.close()
        tmp_path = tmp.name

        # Inferencia con aceleración GPU (device=0)
        results_pose = model_pose(tmp_path, device=0, stream=False, conf=0.5)
        results_depth = model_depth(tmp_path, device=0, imgsz=768, stream=False)

        # Capturar el frame real exacto con OpenCV ANTES
        frame_b64 = ""
        cap = cv2.VideoCapture(tmp_path)
        ret, frame_real = cap.read()
        cap.release()

        orig_h, orig_w = 640, 640
        if ret and frame_real is not None:
            orig_h, orig_w = frame_real.shape[:2]

        keypoints_3d = {}
        person_kpts = []

        for r_pose, r_depth in zip(results_pose, results_depth):
            if r_pose.keypoints is not None and len(r_pose.keypoints.data) > 0:
                kpts_2d = r_pose.keypoints.xy.cpu().numpy()
                depth_map = r_depth.depth.data.cpu().numpy()
                h_depth, w_depth = depth_map.shape

                person_kpts = kpts_2d[0]
                sx = w_depth / orig_w
                sy = h_depth / orig_h

                for idx, (x, y) in enumerate(person_kpts):
                    x_d = int(np.clip(x * sx, 0, w_depth - 1))
                    y_d = int(np.clip(y * sy, 0, h_depth - 1))
                    z_metrico = float(depth_map[y_d, x_d])

                    keypoints_3d[str(idx)] = {
                        "x": float(x),
                        "y": float(y),
                        "z": z_metrico
                    }
                break

        if not keypoints_3d:
            return {"error": "No se detectó sujeto activo o keypoints válidos."}

        if ret and frame_real is not None:
            # Dibujar keypoints sin reescalar
            for pt in person_kpts:
                x_raw, y_raw = int(pt[0]), int(pt[1])
                if 0 <= x_raw < orig_w and 0 <= y_raw < orig_h:
                    cv2.circle(frame_real, (x_raw, y_raw), 8, (0, 255, 0), -1)

            _, buffer = cv2.imencode(".jpg", frame_real, [cv2.IMWRITE_JPEG_QUALITY, 85])
            frame_b64 = base64.b64encode(buffer).decode("utf-8")

        return {
            "keypoints_3d": keypoints_3d,
            "frame_base64": f"data:image/jpeg;base64,{frame_b64}" if frame_b64 else "",
            "desviaciones": []
        }
    finally:
        if os.path.exists(tmp_path):
            try:
                os.remove(tmp_path)
            except Exception:
                pass
        torch.cuda.empty_cache()


### Celda 4b: Carga del Modelo de Embeddings Qwen (2048 dimensiones)

> **Requisito Multimodal:** Carga del modelo  en GPU Nvidia para vectorización densa (2048d) sin saturar hardware local.

In [ ]:
# Celda 4b: Carga del Modelo de Embeddings Qwen (2048 dimensiones)
from sentence_transformers import SentenceTransformer
import torch

print("Cargando Qwen3-VL-Embedding-2B en VRAM...")
# Usamos trust_remote_code=True porque es un modelo de HuggingFace
embedding_model = SentenceTransformer(
    "Qwen/Qwen3-VL-Embedding-2B", 
    device="cuda", 
    trust_remote_code=True,
    model_kwargs={
        "torch_dtype": torch.bfloat16
    }
)
print("✅ Modelo de Embeddings listo.")

### Celda 5: Servidor HTTP Flask y Túnel Público Ngrok

> **Instrucciones para Persistencia del Túnel:**
> 1. Obtén tu token personal gratuito en [dashboard.ngrok.com](https://dashboard.ngrok.com).
> 2. Reemplaza `TU_NGROK_AUTH_TOKEN_AQUI` con tu token.
> 3. Mantén abierta la pestaña de Google Colab en tu navegador para evitar que la sesión entre en reposo.
> 4. Copia la URL pública generada (`https://xxxx.ngrok-free.app`) en tu archivo `.env` local (`COLAB_TUNNEL_URL=https://xxxx.ngrok-free.app`).

In [ ]:
# Celda 5: Servidor Flask y Túnel Ngrok
from flask import Flask, request, jsonify
from pyngrok import ngrok

app = Flask(__name__)

# CONFIGURACIÓN DE NGROK
NGROK_AUTH_TOKEN = "3JBqQghLHManErBLA80aFdKUBYR_5aTVx17Kzhf5gYVCju9mW"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

@app.route("/inferir", methods=["POST"])
def inferir():
    if "file" not in request.files:
        return jsonify({"error": "No file part. Se requiere archivo multipart con clave 'file'."}), 400
    
    file = request.files["file"]
    if file.filename == '':
        return jsonify({"error": "No selected file."}), 400
        
    try:
        video_bytes = file.read()
        resultado = procesar_video_y_fusionar_3d(video_bytes)
        if "error" in resultado:
            return jsonify(resultado), 400
        return jsonify(resultado), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route("/embed", methods=["POST"])
def generar_embeddings():
    data = request.get_json()
    textos = data.get("textos", [])
    if not textos:
        return jsonify({"error": "Se requiere una lista de textos"}), 400
    try:
        import torch
        # --- MICRO-BATCHING EN COLAB ---
        MICRO_BATCH_SIZE = 32
        all_embeddings = []
        for i in range(0, len(textos), MICRO_BATCH_SIZE):
            micro_batch = textos[i : i + MICRO_BATCH_SIZE]
            with torch.no_grad():
                embeddings = embedding_model.encode(
                    micro_batch, 
                    normalize_embeddings=True, 
                    batch_size=MICRO_BATCH_SIZE
                ).tolist()
            all_embeddings.extend(embeddings)
            torch.cuda.empty_cache()
        return jsonify({"embeddings": all_embeddings}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

# Exponer puerto 5000 a través del túnel seguro Ngrok
public_url = ngrok.connect(5000)
print("=" * 64)
print("🚀 SERVIDOR YOLO26 ACTIVO")
print(f"🔗 NGROK URL: {public_url.public_url}")
print(f"👉 Copia esta URL en .env: COLAB_TUNNEL_URL={public_url.public_url}")
print("=" * 64)

app.run(port=5000, debug=False)
